# TC3-42 — Análise de Risco de Data Leakage

**Responsável:** Benito Marculano Ribeiro

**Sprint:** Fase 2

**Jira:** TC3-42

**Objetivo:** revisar as variáveis e o processo de preparação para identificar informações que revelem direta ou indiretamente o target ou dados posteriores ao momento da previsão.

Este notebook parte do catálogo produzido no TC3-40 e das decisões preliminares do TC3-41. O foco aqui não é treinar modelo, mas classificar o risco de vazamento das variáveis candidatas.

## 1. Escopo da análise

A análise de Data Leakage verifica se uma variável poderia fornecer ao modelo informação que não estaria disponível no momento legítimo da predição ou que esteja diretamente ligada à construção do target.

Classificação utilizada:

- **Baixo:** variável aparentemente disponível antes da predição e sem relação direta com o target.
- **Médio:** variável potencialmente útil, mas exige cuidado com encoding, granularidade, temporalidade ou desenho amostral.
- **Alto:** variável com forte suspeita de carregar informação posterior, operacional da avaliação ou diretamente relacionada ao target.
- **Não aplicável:** variável de controle técnico, target ou identificador que não deve entrar como feature.

As conclusões deste notebook devem orientar a seleção final de features e o pré-processamento posterior.

## 2. Leitura do catálogo TC3-40

A fonte inicial é o arquivo `docs/data_dictionary/ml_aluno_catalogo_variaveis.csv`, gerado pelo TC3-40.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 180)


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        catalogo = candidate / "docs" / "data_dictionary" / "ml_aluno_catalogo_variaveis.csv"
        if (candidate / "README.md").exists() and catalogo.exists():
            return candidate
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto com docs/data_dictionary/ml_aluno_catalogo_variaveis.csv.")


project_root = find_project_root()
catalogo_path = project_root / "docs" / "data_dictionary" / "ml_aluno_catalogo_variaveis.csv"

catalogo_ml_aluno = pd.read_csv(catalogo_path)
print(f"Catálogo carregado de: {catalogo_path}")
print(f"Linhas: {catalogo_ml_aluno.shape[0]} | Colunas: {catalogo_ml_aluno.shape[1]}")

catalogo_ml_aluno

## 3. Matriz de risco de leakage

A matriz abaixo classifica cada variável da `ml_aluno` quanto ao risco de Data Leakage e define uma recomendação de uso.

In [ ]:
riscos_leakage = pd.DataFrame([
    {
        "Variável": "id_aluno",
        "risco_leakage": "Não aplicável",
        "recomendacao_tc3_42": "Excluir como feature",
        "justificativa_leakage": "Identificador individual. Pode induzir memorização e não representa característica generalizável para novos casos.",
    },
    {
        "Variável": "id_escola",
        "risco_leakage": "Médio",
        "recomendacao_tc3_42": "Usar com cautela",
        "justificativa_leakage": "Código de escola pode capturar contexto real, mas o uso direto pode gerar memorização por alta cardinalidade. Preferir enriquecimento/agregações.",
    },
    {
        "Variável": "id_municipio",
        "risco_leakage": "Médio",
        "recomendacao_tc3_42": "Usar com cautela",
        "justificativa_leakage": "Código territorial é útil, mas o uso direto pode capturar padrões locais muito específicos. Preferir indicadores municipais e encoding controlado.",
    },
    {
        "Variável": "nome_municipio",
        "risco_leakage": "Não aplicável",
        "recomendacao_tc3_42": "Excluir como feature",
        "justificativa_leakage": "Campo textual redundante em relação a `id_municipio`; não adiciona sinal novo e pode criar inconsistências de codificação.",
    },
    {
        "Variável": "sigla_uf",
        "risco_leakage": "Baixo",
        "recomendacao_tc3_42": "Pode usar",
        "justificativa_leakage": "Informação territorial disponível previamente e sem relação direta com a construção do target.",
    },
    {
        "Variável": "ano_referencia",
        "risco_leakage": "Baixo",
        "recomendacao_tc3_42": "Pode usar",
        "justificativa_leakage": "Variável temporal da observação. Deve ser usada como controle de safra/ano, sem misturar treino e teste temporalmente de forma indevida.",
    },
    {
        "Variável": "serie",
        "risco_leakage": "Baixo",
        "recomendacao_tc3_42": "Pode usar",
        "justificativa_leakage": "Informação escolar disponível antes da avaliação e relevante para contexto educacional.",
    },
    {
        "Variável": "rede",
        "risco_leakage": "Baixo",
        "recomendacao_tc3_42": "Pode usar após encoding",
        "justificativa_leakage": "Informação administrativa disponível previamente. Exige tratamento categórico, mas não aparenta revelar o target.",
    },
    {
        "Variável": "rede_descricao",
        "risco_leakage": "Não aplicável",
        "recomendacao_tc3_42": "Excluir como feature",
        "justificativa_leakage": "Descrição textual redundante com `rede`. Usar as duas duplicaria informação.",
    },
    {
        "Variável": "caderno",
        "risco_leakage": "Médio",
        "recomendacao_tc3_42": "Investigar antes de usar",
        "justificativa_leakage": "Identifica instrumento/caderno da avaliação. Pode estar ligado ao processo de mensuração e não necessariamente ao contexto prévio do aluno.",
    },
    {
        "Variável": "presenca",
        "risco_leakage": "Alto",
        "recomendacao_tc3_42": "Não usar sem validação específica",
        "justificativa_leakage": "Presença na avaliação pode condicionar existência de proficiência e target. É informação operacional do evento avaliado.",
    },
    {
        "Variável": "preenchimento_caderno",
        "risco_leakage": "Alto",
        "recomendacao_tc3_42": "Não usar sem validação específica",
        "justificativa_leakage": "Reflete comportamento/estado durante a avaliação, podendo carregar sinal posterior ou ligado à mensuração do target.",
    },
    {
        "Variável": "proficiencia",
        "risco_leakage": "Alto",
        "recomendacao_tc3_42": "Excluir até comprovação contrária",
        "justificativa_leakage": "Variável de desempenho provavelmente relacionada diretamente à classificação `alfabetizado`. Forte suspeita de leakage por definição do target.",
    },
    {
        "Variável": "peso_aluno",
        "risco_leakage": "Médio",
        "recomendacao_tc3_42": "Não usar como feature; avaliar como peso amostral",
        "justificativa_leakage": "Peso amostral pode refletir desenho da avaliação. Pode ser útil para ponderação, mas não como atributo explicativo do aluno.",
    },
    {
        "Variável": "taxa_alfabetizacao_municipio_ano_anterior",
        "risco_leakage": "Baixo",
        "recomendacao_tc3_42": "Pode usar com validação temporal",
        "justificativa_leakage": "Indicador agregado do ano anterior. A temporalidade sugere baixo risco, desde que confirmado que não usa dados do ano corrente.",
    },
    {
        "Variável": "media_portugues_municipio_ano_anterior",
        "risco_leakage": "Baixo",
        "recomendacao_tc3_42": "Pode usar com validação temporal",
        "justificativa_leakage": "Indicador agregado do ano anterior. Baixo risco se calculado apenas com informações anteriores ao período de predição.",
    },
    {
        "Variável": "dataset_split",
        "risco_leakage": "Não aplicável",
        "recomendacao_tc3_42": "Excluir como feature",
        "justificativa_leakage": "Variável técnica de separação treino/validação/teste. Deve controlar experimentos, nunca entrar no modelo.",
    },
    {
        "Variável": "alfabetizado",
        "risco_leakage": "Não aplicável",
        "recomendacao_tc3_42": "Usar apenas como target",
        "justificativa_leakage": "É a variável alvo. Usá-la como feature seria vazamento direto.",
    },
    {
        "Variável": "processed_at",
        "risco_leakage": "Não aplicável",
        "recomendacao_tc3_42": "Excluir como feature",
        "justificativa_leakage": "Timestamp técnico de processamento da base. Não representa atributo educacional disponível no momento de predição.",
    },
])

analise_leakage = catalogo_ml_aluno.merge(
    riscos_leakage,
    on="Variável",
    how="left",
    validate="one_to_one",
)

analise_leakage

## 4. Resumo dos riscos

In [ ]:
resumo_risco = (
    analise_leakage
    .groupby("risco_leakage")
    .agg(
        quantidade=("Variável", "count"),
        variaveis=("Variável", lambda x: ", ".join(x)),
    )
    .reset_index()
)

resumo_risco

## 5. Variáveis recomendadas para uso imediato

Variáveis classificadas com risco baixo podem seguir para modelagem/preparação, respeitando tratamentos posteriores como encoding e imputação.

In [ ]:
variaveis_baixo_risco = analise_leakage.query("risco_leakage == 'Baixo'")
variaveis_baixo_risco[["Variável", "Descrição", "Categoria", "recomendacao_tc3_42"]]

## 6. Variáveis bloqueadas ou pendentes

Variáveis com risco médio ou alto não devem entrar automaticamente como features. Elas exigem decisão explícita da equipe ou tratamento alternativo.

In [ ]:
variaveis_atencao = analise_leakage.query("risco_leakage in ['Médio', 'Alto']")
variaveis_atencao[["Variável", "Descrição", "Categoria", "risco_leakage", "recomendacao_tc3_42", "justificativa_leakage"]]

## 7. Parecer técnico

A análise indica que o maior risco de Data Leakage está nas variáveis ligadas diretamente ao processo de avaliação e mensuração do desempenho: `proficiencia`, `presenca` e `preenchimento_caderno`.

Para uma primeira modelagem mais conservadora, recomenda-se:

- usar variáveis territoriais, escolares e administrativas de baixo risco;
- usar indicadores municipais de ano anterior apenas após confirmação da temporalidade de cálculo;
- não usar `proficiencia` como feature enquanto não houver comprovação de que ela não define diretamente o target;
- tratar `peso_aluno` como possível peso amostral, não como variável explicativa;
- manter `dataset_split`, `alfabetizado` e `processed_at` fora do conjunto de features.

Esta classificação deve alimentar as próximas etapas de preparação dos dados, pipeline de modelagem e documentação final das limitações do projeto.